# 06 - 权重初始化

## 学习目标

1. 理解 Xavier 初始化（均匀分布和正态分布）
2. 理解 Kaiming 初始化（均匀分布和正态分布）
3. 掌握其他初始化方法
4. 学会使用 `calculate_gain()` 计算增益
5. 使用 `apply()` 按层类型初始化权重
6. 对比默认初始化与合理初始化对训练的影响

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.init as init

## 1. 为什么需要合理的权重初始化？

如果权重初始化不当：
- **全部为零**：所有神经元学到相同的东西（对称性问题）
- **太大**：梯度爆炸，训练不稳定
- **太小**：梯度消失，学不到东西

合理的初始化目标：**保持前向传播和反向传播时信号的方差不变**。

In [ ]:
# 不好的初始化：方差在层间剧烈变化
torch.manual_seed(42)

x = torch.randn(256, 784)  # 256 个样本，784 维
print(f"输入方差: {x.var().item():.4f}")

# 模拟 5 层网络（大权重）
print("\n大权重初始化 (std=1.0):")
h = x
for i in range(5):
    w = torch.randn(784, 784)  # std=1.0, 太大
    h = torch.relu(h @ w)
    print(f"  第 {i+1} 层输出方差: {h.var().item():.4f}")

# 模拟 5 层网络（小权重）
print("\n小权重初始化 (std=0.01):")
h = x
for i in range(5):
    w = torch.randn(784, 784) * 0.01  # std=0.01, 太小
    h = torch.relu(h @ w)
    print(f"  第 {i+1} 层输出方差: {h.var().item():.10f}")

**观察：** 大权重导致方差爆炸，小权重导致方差趋近于零。

## 2. Xavier 初始化

**适用于**：Sigmoid / Tanh 等饱和激活函数。

核心思想：让每层输出的方差与输入的方差相同。

### Xavier Uniform
$$W \sim U\left(-\sqrt{\frac{6}{n_{in} + n_{out}}}, \sqrt{\frac{6}{n_{in} + n_{out}}}\right)$$

### Xavier Normal
$$W \sim N\left(0, \frac{2}{n_{in} + n_{out}}\right)$$

In [ ]:
# Xavier 初始化
linear = nn.Linear(784, 256)

# Xavier Uniform
init.xavier_uniform_(linear.weight)
print("Xavier Uniform:")
print(f"  均值: {linear.weight.mean().item():.6f}")
print(f"  标准差: {linear.weight.std().item():.6f}")
print(f"  范围: [{linear.weight.min().item():.4f}, {linear.weight.max().item():.4f}]")

# Xavier Normal
init.xavier_normal_(linear.weight)
print("\nXavier Normal:")
print(f"  均值: {linear.weight.mean().item():.6f}")
print(f"  标准差: {linear.weight.std().item():.6f}")

In [ ]:
# 验证 Xavier + Tanh: 方差保持稳定
torch.manual_seed(42)
x = torch.randn(256, 784)

print("Xavier Normal + Tanh:")
print(f"输入方差: {x.var().item():.4f}")
h = x
for i in range(5):
    w = torch.empty(784, 784)
    init.xavier_normal_(w)
    h = torch.tanh(h @ w)
    print(f"  第 {i+1} 层输出方差: {h.var().item():.4f}")

## 3. Kaiming 初始化

**适用于**：ReLU / LeakyReLU 等非饱和激活函数。

Xavier 假设激活函数是线性的，对 ReLU 不适用（ReLU 会砍掉一半值）。Kaiming 初始化针对 ReLU 做了修正。

### Kaiming Uniform
$$W \sim U\left(-\sqrt{\frac{6}{(1+a^2) \times n}}, \sqrt{\frac{6}{(1+a^2) \times n}}\right)$$

### Kaiming Normal
$$W \sim N\left(0, \frac{2}{(1+a^2) \times n}\right)$$

其中 `a` 是 LeakyReLU 的负斜率（ReLU 时 a=0），`n` 取决于 `mode`。

In [ ]:
# Kaiming 初始化
linear = nn.Linear(784, 256)

# Kaiming Uniform (fan_in)
init.kaiming_uniform_(linear.weight, mode='fan_in', nonlinearity='relu')
print("Kaiming Uniform (fan_in, relu):")
print(f"  均值: {linear.weight.mean().item():.6f}")
print(f"  标准差: {linear.weight.std().item():.6f}")

# Kaiming Normal (fan_out)
init.kaiming_normal_(linear.weight, mode='fan_out', nonlinearity='relu')
print("\nKaiming Normal (fan_out, relu):")
print(f"  均值: {linear.weight.mean().item():.6f}")
print(f"  标准差: {linear.weight.std().item():.6f}")

# 适用于 LeakyReLU
init.kaiming_normal_(linear.weight, a=0.1, mode='fan_in', nonlinearity='leaky_relu')
print("\nKaiming Normal (fan_in, leaky_relu a=0.1):")
print(f"  均值: {linear.weight.mean().item():.6f}")
print(f"  标准差: {linear.weight.std().item():.6f}")

In [ ]:
# 验证 Kaiming + ReLU: 方差保持稳定
torch.manual_seed(42)
x = torch.randn(256, 784)

print("Kaiming Normal + ReLU:")
print(f"输入方差: {x.var().item():.4f}")
h = x
for i in range(5):
    w = torch.empty(784, 784)
    init.kaiming_normal_(w, mode='fan_in', nonlinearity='relu')
    h = torch.relu(h @ w)
    print(f"  第 {i+1} 层输出方差: {h.var().item():.4f}")

**`mode` 参数：**

- `fan_in`（默认）：保持前向传播的方差稳定
- `fan_out`：保持反向传播的方差稳定

实践中两者差别不大，`fan_in` 更常用。

## 4. 其他初始化方法

In [ ]:
w = torch.empty(3, 3)

# uniform_: 均匀分布
init.uniform_(w, a=-1.0, b=1.0)
print(f"uniform_(-1, 1):\n{w}\n")

# normal_: 正态分布
init.normal_(w, mean=0.0, std=0.1)
print(f"normal_(0, 0.1):\n{w}\n")

# constant_: 常数
init.constant_(w, val=0.5)
print(f"constant_(0.5):\n{w}\n")

# zeros_: 全零
init.zeros_(w)
print(f"zeros_:\n{w}\n")

# ones_: 全一
init.ones_(w)
print(f"ones_:\n{w}")

In [ ]:
# eye_: 单位矩阵（仅适用于 2D 张量）
w = torch.empty(3, 3)
init.eye_(w)
print(f"eye_:\n{w}\n")

# orthogonal_: 正交初始化
w = torch.empty(3, 3)
init.orthogonal_(w)
print(f"orthogonal_:\n{w}")
# 验证正交性: W @ W^T ≈ I
print(f"\nW @ W^T (应接近单位矩阵):\n{w @ w.T}")

**各方法适用场景：**

| 方法 | 适用场景 |
|------|----------|
| `xavier_*` | Sigmoid, Tanh 激活函数 |
| `kaiming_*` | ReLU, LeakyReLU 激活函数 |
| `orthogonal_` | RNN / LSTM |
| `zeros_` | bias 初始化 |
| `ones_` | BatchNorm 的 weight |
| `constant_` | 特定值初始化 |

## 5. calculate_gain() - 计算增益

`nn.init.calculate_gain(nonlinearity)` 返回激活函数的推荐增益值，用于调整初始化的标准差。

In [ ]:
# 不同激活函数的增益值
activations = [
    ('linear', {}),
    ('sigmoid', {}),
    ('tanh', {}),
    ('relu', {}),
    ('leaky_relu', {'param': 0.1}),
    ('leaky_relu', {'param': 0.2}),
]

print("激活函数的 gain 值:")
for name, kwargs in activations:
    gain = init.calculate_gain(name, **kwargs)
    param_str = f" (slope={kwargs.get('param', '')})" if kwargs else ""
    print(f"  {name:15s}{param_str:15s}  gain = {gain:.4f}")

In [ ]:
# 在 Xavier 初始化中使用 gain
linear = nn.Linear(784, 256)

# 针对 ReLU 的 Xavier 初始化（手动传入 gain）
gain = init.calculate_gain('relu')
init.xavier_uniform_(linear.weight, gain=gain)
print(f"Xavier + ReLU gain ({gain:.4f}):")
print(f"  标准差: {linear.weight.std().item():.6f}")

## 6. 实战：使用 apply() 按层类型初始化

在实际项目中，最常见的做法是定义一个初始化函数，然后用 `model.apply()` 应用到所有层。

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(64, 10)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


def initialize_weights(m):
    """按层类型进行权重初始化。"""
    if isinstance(m, nn.Conv2d):
        # 卷积层: Kaiming 初始化
        init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        if m.bias is not None:
            init.zeros_(m.bias)
    elif isinstance(m, nn.BatchNorm2d):
        # BatchNorm: weight=1, bias=0
        init.ones_(m.weight)
        init.zeros_(m.bias)
    elif isinstance(m, nn.Linear):
        # 全连接层: Xavier 初始化
        init.xavier_uniform_(m.weight)
        if m.bias is not None:
            init.zeros_(m.bias)


# 创建模型并初始化
model = CNN()
model.apply(initialize_weights)

# 检查初始化结果
print("初始化后的参数统计:")
for name, param in model.named_parameters():
    if 'weight' in name:
        print(f"  {name:30s}  mean={param.mean().item():+.6f}  std={param.std().item():.6f}")

## 7. 对比：默认初始化 vs 合理初始化

让我们通过一个简单的训练实验来看初始化的影响。

In [ ]:
import torch.optim as optim

# 生成简单的分类数据
torch.manual_seed(42)
num_samples = 256
x_data = torch.randn(num_samples, 1, 8, 8)
y_data = torch.randint(0, 10, (num_samples,))


def train_model(model, x_data, y_data, epochs=50):
    """训练模型并记录 loss。"""
    optimizer = optim.SGD(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()
    losses = []

    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        output = model(x_data)
        loss = criterion(output, y_data)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    return losses


# 方案 1: 默认初始化
torch.manual_seed(42)
model_default = CNN()
losses_default = train_model(model_default, x_data, y_data)

# 方案 2: Kaiming 初始化
torch.manual_seed(42)
model_kaiming = CNN()
model_kaiming.apply(initialize_weights)
losses_kaiming = train_model(model_kaiming, x_data, y_data)

# 方案 3: 不好的初始化（太大）
def bad_init(m):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        init.normal_(m.weight, mean=0, std=1.0)  # 标准差太大

torch.manual_seed(42)
model_bad = CNN()
model_bad.apply(bad_init)
losses_bad = train_model(model_bad, x_data, y_data)

# 对比结果
print(f"{'Epoch':>6s}  {'默认':>10s}  {'Kaiming':>10s}  {'太大':>10s}")
print("-" * 42)
for epoch in [0, 9, 19, 29, 39, 49]:
    print(f"{epoch+1:>6d}  {losses_default[epoch]:>10.4f}  {losses_kaiming[epoch]:>10.4f}  {losses_bad[epoch]:>10.4f}")

**观察：**

- 默认初始化和 Kaiming 初始化效果通常都不错（PyTorch 内置层的默认初始化已经比较合理）
- 太大的初始化会导致训练不稳定（loss 可能 NaN 或收敛慢）
- 在深层网络中，合理初始化的优势更加明显

## 8. 练习题

### 练习 1：实现自定义初始化函数

编写一个初始化函数，对不同层使用不同策略：
- Conv2d: Kaiming Normal (fan_in, relu)
- Linear: Xavier Uniform
- BatchNorm2d: weight=1, bias=0
- 所有 bias: 0

In [ ]:
def custom_init(m):
    """自定义权重初始化。"""
    # TODO: 根据层类型选择初始化方式
    pass


# 测试代码
# model = CNN()
# model.apply(custom_init)
# for name, param in model.named_parameters():
#     if 'weight' in name:
#         print(f"{name}: mean={param.mean().item():.6f}, std={param.std().item():.6f}")

### 练习 2：验证 Xavier vs Kaiming

构建一个 10 层的全连接网络，分别使用 Xavier 和 Kaiming 初始化 + ReLU，观察各层输出方差的变化。

In [ ]:
# TODO: 构建 10 层网络，对比两种初始化
# 提示: 用 torch.relu() 作为激活函数

# dim = 512
# x = torch.randn(256, dim)
#
# print("Xavier Normal + ReLU:")
# h = x
# for i in range(10):
#     w = torch.empty(dim, dim)
#     init.xavier_normal_(w)
#     h = torch.relu(h @ w)
#     print(f"  第 {i+1} 层方差: {h.var().item():.6f}")
#
# print("\nKaiming Normal + ReLU:")
# h = x
# for i in range(10):
#     w = torch.empty(dim, dim)
#     init.kaiming_normal_(w, mode='fan_in', nonlinearity='relu')
#     h = torch.relu(h @ w)
#     print(f"  第 {i+1} 层方差: {h.var().item():.6f}")

## 9. 小结

### 初始化方法速查

| 方法 | 适用激活函数 | 函数 |
|------|------------|------|
| Xavier Uniform | Sigmoid, Tanh | `init.xavier_uniform_(w)` |
| Xavier Normal | Sigmoid, Tanh | `init.xavier_normal_(w)` |
| Kaiming Uniform | ReLU, LeakyReLU | `init.kaiming_uniform_(w)` |
| Kaiming Normal | ReLU, LeakyReLU | `init.kaiming_normal_(w)` |
| Orthogonal | RNN/LSTM | `init.orthogonal_(w)` |

### 实践建议

1. **CNN + ReLU** → Kaiming 初始化
2. **Transformer** → Xavier 初始化（内部通常用 Tanh/GELU）
3. **BatchNorm** → weight=1, bias=0
4. **Bias** → 通常初始化为 0
5. **PyTorch 默认初始化**已经比较合理，一般不需要手动调整

### 标准初始化模板

```python
def initialize_weights(m):
    if isinstance(m, nn.Conv2d):
        init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        if m.bias is not None:
            init.zeros_(m.bias)
    elif isinstance(m, nn.BatchNorm2d):
        init.ones_(m.weight)
        init.zeros_(m.bias)
    elif isinstance(m, nn.Linear):
        init.xavier_uniform_(m.weight)
        init.zeros_(m.bias)

model.apply(initialize_weights)
```

### 关键要点

1. 权重初始化的目标是**保持前向/反向传播时的方差稳定**
2. Xavier 适配线性/饱和激活，Kaiming 适配 ReLU 系列
3. `mode='fan_in'` 保持前向方差，`mode='fan_out'` 保持反向方差
4. `calculate_gain()` 返回激活函数的增益值
5. 使用 `model.apply(init_fn)` 统一初始化所有层